<a href="https://colab.research.google.com/github/AlfredV12/ML-Sample/blob/main/CredisCardNew.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AlfredV12/ML_Sample/blob/main/CreditCardNew.ipynb)

# ==============================================================================
# PROJECT: LOAN APPROVAL PREDICTION & CLASS IMBALANCE OPTIMIZATION
# Framework: Data Preparation -> Feature Engineering -> SMOTE -> Model Evaluation
# ==============================================================================

In [ ]:
# ==============================================================================
# PROJECT: CREDIT RISK DEFAULT SCORING & CLASS IMBALANCE OPTIMIZATION
# Framework: Data Preparation -> Feature Engineering -> SMOTE -> Model Evaluation
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Scikit-Learn Modules
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    recall_score
)

# Imbalanced-Learn Module for SMOTE
from imblearn.over_sampling import SMOTE

# Configuration
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 5)

print("✅ All dependencies loaded successfully!")

# ------------------------------------------------------------------------------
# STEP 1: LOAD THE DATA (TRAIN AND TEST)
# ------------------------------------------------------------------------------
print("\nအ Loading datasets from local files...")

# Correctly load training data with labels, and test data
df_train = pd.read_csv("/content/cs-training.csv", index_col=0)

# ------------------------------------------------------------------------------
# STEP 2: DATA PREPARATION & FEATURE ENGINEERING
# ------------------------------------------------------------------------------
def preprocess_and_engineer(data):
    """Applies missing value imputation and feature engineering."""
    df_clean = data.copy()

    # Imputation
    df_clean['MonthlyIncome'] = df_clean['MonthlyIncome'].fillna(df_clean['MonthlyIncome'].median())
    df_clean['NumberOfDependents'] = df_clean['NumberOfDependents'].fillna(df_clean['NumberOfDependents'].mode()[0])

    # Outlier Capping
    util_cap = 10.0
    df_clean['RevolvingUtilizationOfUnsecuredLines'] = np.where(
        df_clean['RevolvingUtilizationOfUnsecuredLines'] > util_cap,
        util_cap,
        df_clean['RevolvingUtilizationOfUnsecuredLines']
    )

    # Feature Engineering
    df_clean['TotalDelinquencyEvents'] = (
        df_clean['NumberOfTime30-59DaysPastDueNotWorse'] +
        df_clean['NumberOfTime60-89DaysPastDueNotWorse'] +
        df_clean['NumberOfTimes90DaysLate']
    )
    df_clean['IncomePerCapita'] = df_clean['MonthlyIncome'] / (df_clean['NumberOfDependents'] + 1)
    df_clean['EstimatedMonthlyDebtUSD'] = df_clean['DebtRatio'] * df_clean['MonthlyIncome']
    df_clean['SevereDelinquencyRatio'] = np.where(
        df_clean['TotalDelinquencyEvents'] > 0,
        df_clean['NumberOfTimes90DaysLate'] / df_clean['TotalDelinquencyEvents'],
        0
    )
    return df_clean

# Apply processing to training data
df_train_processed = preprocess_and_engineer(df_train)

# Separate Features (X) and Target (y) from labeled training set
X = df_train_processed.drop(columns=['SeriousDlqin2yrs'])
y = df_train_processed['SeriousDlqin2yrs']

# Create a labeled validation set from our training data since cs-test.csv has no labels
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ------------------------------------------------------------------------------
# STEP 3: CLASS IMBALANCE HANDLING (SMOTE)
# ------------------------------------------------------------------------------
print("\n⚖  Applying SMOTE to balance the training data...")
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

# ------------------------------------------------------------------------------
# STEP 4: MODEL TRAINING
# ------------------------------------------------------------------------------
print("\nဢ Training Models...")

# Baseline Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_resampled, y_train_resampled)
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

# Advanced Random Forest
rf_model = RandomForestClassifier(
    n_estimators=150, max_depth=12, min_samples_split=10,
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

# ------------------------------------------------------------------------------
# STEP 5: EVALUATION ON TEST SET
# ------------------------------------------------------------------------------
print("\n" + "="*50)
print(" RANDOM FOREST CLASSIFICATION REPORT (Test Data)")
print("="*50)
print(classification_report(y_test, y_pred_rf, target_names=['Non-Default (0)', 'Default (1)']))

rf_auc = roc_auc_score(y_test, y_prob_rf)
lr_auc = roc_auc_score(y_test, y_prob_lr)

print(f"ဨ Model ROC-AUC Benchmarks:")
print(f" - Baseline Logistic Regression AUC: {lr_auc:.4f}")
print(f" - Advanced Random Forest AUC:      {rf_auc:.4f}")

# Threshold Optimization
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob_rf)
opt_idx = np.argmin(np.abs(recalls - 0.75)) # Target Recall >= 0.75
optimal_threshold = thresholds[opt_idx]
y_pred_optimal = (y_prob_rf >= optimal_threshold).astype(int)

# --- VISUAL DIAGNOSTICS DASHBOARD ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Confusion Matrix
cm = confusion_matrix(y_test, y_pred_optimal)
sns.heatmap(cm, annot=True, fmt=',d', cmap='Reds', ax=axes[0],
            xticklabels=['Non-Default', 'Default'],
            yticklabels=['Non-Default', 'Default'], cbar=False)
axes[0].set_title(f'Confusion Matrix\n(Optimized Threshold = {optimal_threshold:.3f})', fontweight='bold')
axes[0].set_xlabel('Predicted Risk Class')
axes[0].set_ylabel('True Risk Class')

# 2. ROC Curve
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)

axes[1].plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {lr_auc:.2f})', linestyle='--', color='tab:blue')
axes[1].plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {rf_auc:.2f})', linewidth=2.5, color='tab:red')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[1].set_title('ROC Curve Benchmarking', fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc='lower right')

# 3. Feature Importance
importances = pd.Series(rf_model.feature_importances_, index=X_train.columns).sort_values(ascending=True)
importances.tail(8).plot(kind='barh', ax=axes[2], color='navy')
axes[2].set_title('Top 8 Risk Drivers (Feature Importance)', fontweight='bold')
axes[2].set_xlabel('Relative Gini Importance Score')

plt.tight_layout()
plt.show()